# 03 — Kalman R1C1

Filtre **à la main** (`run_kalman` / `filter_r1c1`). Cycle : prédire avec u,
innover e = y - Tchapeau, corriger.

Sur un R1C1 dont on connaît l'état vrai, le filtre doit battre la mesure brute.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from basic_mpc.identification.pem import filter_r1c1
from basic_mpc.models.r1c1 import R1C1Params, simulate_r1c1

params = R1C1Params(
    a=0.96, g_solar=2e-5, g_heating=4e-3,
    process_noise_std=0.03, sensor_noise_std=0.25,
)
n = 400
hours = np.arange(n) * 0.25
t_ext = 8.0 + 6.0 * np.sin(2 * np.pi * hours / 24)
solar = np.clip(np.sin(2 * np.pi * (hours - 6) / 24), 0, None) * 2000
heating = np.where((hours % 24 < 8) | (hours % 24 > 20), 20.0, 0.0)
x_true, y = simulate_r1c1(params, t_ext, solar, heating, x0=18.0, seed=2)
u = np.column_stack([t_ext, solar, heating])
res = filter_r1c1(y, u, params)
x_hat = res.x_filt[:, 0]
rmse_f = float(np.sqrt(np.mean((x_hat - x_true) ** 2)))
rmse_y = float(np.sqrt(np.mean((y - x_true) ** 2)))
print(f"RMSE filtre {rmse_f:.3f} °C  vs  mesure {rmse_y:.3f} °C")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(hours, x_true, color="#2c2416", label="état vrai")
ax.plot(hours, y, color="#8a7e6e", lw=0.7, alpha=0.8, label="y")
ax.plot(hours, x_hat, color="#3d6b6b", label="Kalman")
ax.set_xlabel("heures")
ax.set_ylabel("°C")
ax.set_title("Le filtre lisse le capteur ; il ne crée pas d'état caché (R1C1)")
ax.legend(frameon=False)
plt.show()